In [1]:
import os
from pathlib import Path

DATA_ROOT = Path.home() / "mne_data"

os.environ["MNE_DATA"] = str(DATA_ROOT)
os.environ["MNE_DATASETS_BNCI_PATH"] = str(DATA_ROOT)
os.environ["MNE_DATASETS_ZHOU2016_PATH"] = str(DATA_ROOT)
os.environ["MOABB_DOWNLOAD_PROVIDER"] = "upstream"

EXPERIMENT_DIR = Path("/Users/liza/Projects/vkr/mi-ood")
CACHE_DIR = EXPERIMENT_DIR / "cache"
RESULTS_DIR = EXPERIMENT_DIR / "results"

print(EXPERIMENT_DIR)

/Users/liza/Projects/vkr/mi-ood


In [2]:
from pipelines import mi_pipelines, ood_pipelines
from processing import *

/Users/liza/Projects/bci-motor-imagery/.venv/lib/python3.11/site-packages/optuna/integration/sklearn.py:14: FutureWarning: `optuna.integration.sklearn` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `optuna_integration.sklearn` instead.
  optuna_warn(f"{msg} Use `optuna_integration.sklearn` instead.", FutureWarning)
/Users/liza/Projects/bci-motor-imagery/.venv/lib/python3.11/site-packages/braindecode/models/eegpt.py:497: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = mne.channels.make_standard_montage("standard_1020")
/Users/liza/Projects/bci-motor-imagery/.venv/lib/python3.11/site-packages/braindecode/models/eegpt.py:1452: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = make_standard_montage("standard_1020")


In [3]:
experiment = "cross-session_7-channels"
COMMON_CHANNELS = ["FC3", "FCz", "FC4", "C3", "C4", "CP3", "CP4"]

datasets = [ "TTK", "Zhou2016", "BNCI2014_001", "Wang2025", "Yang2025_3C", "Zhou2020" ]


In [4]:
for pipeline in [*mi_pipelines.values(), *ood_pipelines.values()]:
    channel_params = {key: tuple(COMMON_CHANNELS) for key in pipeline.get_params(deep=True) if key.split("__")[-1] == "channel_names"}
    pipeline.set_params(**channel_params)

MI_METHODS = ["ACM+TS+SVM", "FBCSP+ElasticNet", "ShallowConvNet", "EEGNet", "Microstates+GB", "VAR+shLDA", "Log-bandpower+shLDA"]
OOD_METHODS = ["ACM+TS+OCSVM", "FBCSP+IsolationForest", "ShallowConvNet+kNN", "EEGNet+TempDens", "Log-bandpower+Mahalanobis", "Microstates+LOF", "VAR+Mahalanobis"]

In [5]:
for dataset in datasets:
    print( dataset )
    subjects = get_subjects(dataset) 

    for subject in subjects:
        data = None

        exp_type = "mi"
        for method, pipeline in mi_pipelines.items():
            if method not in MI_METHODS:
                continue
            result = {"dataset": dataset, "subject": subject, "experiment": experiment, "experiment_type": exp_type, "method": method}
            if result_path(result).exists():
                continue
            if data is None:
                data = load_subject_data(dataset, subject, channels=COMMON_CHANNELS)
            train_data, val_data, test_data = data
            result["MI_BA"], result["MI_AUROC"] = MI_metrics(train_data, test_data, pipeline)
            save_result(result)
            print(f"{dataset} s{subject} {method}: {result['MI_BA']:.3f} {result['MI_AUROC']:.3f}")

        exp_type = "ood"
        data = None
        for method, pipeline in ood_pipelines.items():
            if method not in OOD_METHODS:
                continue
            result = {"dataset": dataset, "subject": subject, "experiment": experiment, "experiment_type": exp_type, "method": method}
            if result_path(result).exists():
                continue
            if data is None:
                ood=("feet",)
                if dataset == "TTK":
                    ood = ("left_foot", "right_foot")
                data = load_subject_data(dataset, subject, channels=COMMON_CHANNELS, ood_classes=ood )
                train_data, val_data, test_data = data
            model = train(train_data, pipeline)

            result["OOD_AUROC"] = OOD_AUROC(test_data, model)
            save_result(result)
            print(f"{dataset} s{subject} {method}: {result['OOD_AUROC']:.3f}")

    display_summary_by_subject(dataset, experiment, experiment_type="mi", metric="MI_BA")
    display_summary_by_subject(dataset, experiment, experiment_type="mi", metric="MI_AUROC")
    display_summary_by_subject(dataset, experiment, experiment_type="ood", metric="OOD_AUROC")


TTK


,MI_BA,n,sub-01,sub-03,sub-04,sub-05,sub-06,sub-07
method,,,,,,,,
EEGNet,0.633,6,0.740,0.620,0.600,0.600,0.716,0.520
FBCSP+ElasticNet,0.626,6,0.890,0.530,0.540,0.530,0.695,0.570
ACM+TS+SVM,0.624,6,0.840,0.600,0.500,0.560,0.746,0.500
Log-bandpower+shLDA,0.601,6,0.730,0.600,0.540,0.540,0.629,0.570
Microstates+GB,0.593,6,0.610,0.580,0.530,0.600,0.628,0.610
ShallowConvNet,0.589,6,0.710,0.630,0.500,0.490,0.637,0.570
VAR+shLDA,0.578,6,0.640,0.610,0.530,0.550,0.637,0.500


,MI_AUROC,n,sub-01,sub-03,sub-04,sub-05,sub-06,sub-07
method,,,,,,,,
ACM+TS+SVM,0.731,6,0.922,0.673,0.626,0.646,0.864,0.652
FBCSP+ElasticNet,0.708,6,0.988,0.574,0.711,0.504,0.844,0.631
EEGNet,0.686,6,0.926,0.597,0.608,0.626,0.818,0.544
Log-bandpower+shLDA,0.652,6,0.848,0.568,0.593,0.527,0.791,0.587
ShallowConvNet,0.639,6,0.888,0.632,0.528,0.514,0.683,0.592
VAR+shLDA,0.623,6,0.700,0.612,0.685,0.580,0.648,0.515
Microstates+GB,0.611,6,0.732,0.542,0.504,0.614,0.652,0.622


,OOD_AUROC,n,sub-01,sub-03,sub-04,sub-05,sub-06,sub-07
method,,,,,,,,
FBCSP+IsolationForest,0.780,6,0.858,0.825,0.671,0.778,0.856,0.690
ShallowConvNet+kNN,0.779,6,0.906,0.684,0.758,0.869,0.813,0.644
EEGNet+TempDens,0.757,6,0.877,0.690,0.690,0.719,0.828,0.739
Log-bandpower+Mahalanobis,0.627,6,0.660,0.738,0.570,0.656,0.600,0.541
VAR+Mahalanobis,0.626,6,0.635,0.717,0.604,0.561,0.596,0.647
ACM+TS+OCSVM,0.571,6,0.568,0.548,0.535,0.614,0.614,0.544
Microstates+LOF,0.565,6,0.669,0.544,0.575,0.657,0.504,0.440


Zhou2016


,MI_BA,n,sub-01,sub-02,sub-03,sub-04
method,,,,,,
EEGNet,0.873,4,0.840,0.822,0.910,0.920
FBCSP+ElasticNet,0.821,4,0.750,0.722,0.890,0.920
ShallowConvNet,0.820,4,0.740,0.711,0.910,0.920
Microstates+GB,0.767,4,0.680,0.667,0.810,0.910
ACM+TS+SVM,0.741,4,0.600,0.633,0.860,0.870
Log-bandpower+shLDA,0.734,4,0.680,0.667,0.880,0.710
VAR+shLDA,0.663,4,0.670,0.633,0.610,0.740


... (14 outputs left out)

Zhou2020


,MI_BA,n,sub-01,sub-02,sub-03,sub-04,sub-05,sub-06,sub-07,sub-08,sub-09,sub-10,sub-11,sub-12,sub-13,sub-14,sub-15,sub-16,sub-17,sub-18,sub-19,sub-20
method,,,,,,,,,,,,,,,,,,,,,,
ShallowConvNet,0.579,20,0.517,0.517,0.667,0.550,0.483,0.683,0.883,0.500,0.525,0.567,0.625,0.492,0.583,0.617,0.775,0.508,0.492,0.550,0.517,0.533
EEGNet,0.576,20,0.492,0.483,0.667,0.500,0.517,0.750,0.850,0.550,0.508,0.492,0.650,0.475,0.533,0.617,0.825,0.550,0.483,0.525,0.533,0.517
FBCSP+ElasticNet,0.562,20,0.500,0.542,0.617,0.433,0.567,0.558,0.792,0.458,0.558,0.525,0.708,0.492,0.500,0.600,0.583,0.533,0.550,0.617,0.608,0.492
Log-bandpower+shLDA,0.554,20,0.542,0.508,0.633,0.467,0.475,0.533,0.717,0.492,0.517,0.558,0.558,0.542,0.575,0.675,0.642,0.500,0.517,0.567,0.533,0.525
Microstates+GB,0.550,20,0.600,0.583,0.642,0.483,0.500,0.675,0.733,0.517,0.458,0.442,0.475,0.567,0.575,0.483,0.683,0.442,0.550,0.508,0.517,0.567
ACM+TS+SVM,0.530,20,0.500,0.475,0.658,0.492,0.500,0.500,0.658,0.500,0.558,0.500,0.525,0.500,0.500,0.642,0.517,0.500,0.500,0.567,0.508,0.500
VAR+shLDA,0.526,20,0.408,0.517,0.492,0.467,0.500,0.600,0.650,0.450,0.508,0.542,0.508,0.458,0.550,0.575,0.617,0.500,0.475,0.558,0.567,0.575


,MI_AUROC,n,sub-01,sub-02,sub-03,sub-04,sub-05,sub-06,sub-07,sub-08,sub-09,sub-10,sub-11,sub-12,sub-13,sub-14,sub-15,sub-16,sub-17,sub-18,sub-19,sub-20
method,,,,,,,,,,,,,,,,,,,,,,
ShallowConvNet,0.612,20,0.520,0.461,0.712,0.548,0.535,0.843,0.938,0.511,0.564,0.539,0.612,0.568,0.608,0.680,0.869,0.528,0.493,0.599,0.537,0.573
EEGNet,0.605,20,0.506,0.476,0.769,0.467,0.529,0.849,0.939,0.553,0.535,0.523,0.658,0.522,0.593,0.639,0.895,0.572,0.486,0.540,0.524,0.528
ACM+TS+SVM,0.602,20,0.436,0.507,0.751,0.416,0.490,0.784,0.804,0.489,0.610,0.463,0.612,0.632,0.520,0.683,0.832,0.512,0.493,0.664,0.681,0.664
FBCSP+ElasticNet,0.602,20,0.584,0.550,0.696,0.413,0.563,0.771,0.878,0.459,0.608,0.580,0.820,0.476,0.524,0.683,0.815,0.491,0.475,0.662,0.584,0.408
Microstates+GB,0.579,20,0.653,0.557,0.696,0.515,0.548,0.672,0.838,0.525,0.454,0.439,0.469,0.526,0.624,0.531,0.786,0.426,0.578,0.473,0.553,0.714
Log-bandpower+shLDA,0.566,20,0.492,0.480,0.657,0.432,0.470,0.709,0.804,0.475,0.535,0.551,0.570,0.530,0.584,0.688,0.715,0.464,0.512,0.588,0.516,0.541
VAR+shLDA,0.557,20,0.427,0.536,0.556,0.483,0.508,0.754,0.762,0.430,0.563,0.518,0.538,0.489,0.602,0.635,0.642,0.502,0.355,0.618,0.577,0.645


,OOD_AUROC,n,sub-01,sub-02,sub-03,sub-04,sub-05,sub-06,sub-07,sub-08,sub-09,sub-10,sub-11,sub-12,sub-13,sub-14,sub-15,sub-16,sub-17,sub-18,sub-19,sub-20
method,,,,,,,,,,,,,,,,,,,,,,
EEGNet+TempDens,0.615,20,0.489,0.523,0.579,0.618,0.522,0.932,0.712,0.511,0.640,0.483,0.589,0.619,0.538,0.820,0.917,0.647,0.460,0.593,0.403,0.710
FBCSP+IsolationForest,0.600,20,0.518,0.433,0.619,0.516,0.697,0.887,0.614,0.496,0.659,0.481,0.632,0.541,0.553,0.738,0.841,0.635,0.505,0.612,0.393,0.636
ShallowConvNet+kNN,0.578,20,0.515,0.549,0.582,0.603,0.599,0.902,0.741,0.480,0.566,0.492,0.625,0.573,0.573,0.817,0.242,0.655,0.537,0.664,0.424,0.427
VAR+Mahalanobis,0.546,20,0.556,0.479,0.556,0.515,0.630,0.493,0.553,0.498,0.587,0.517,0.560,0.561,0.510,0.665,0.654,0.495,0.479,0.598,0.460,0.559
Log-bandpower+Mahalanobis,0.539,20,0.528,0.441,0.525,0.479,0.566,0.418,0.523,0.480,0.736,0.493,0.624,0.519,0.485,0.661,0.638,0.573,0.454,0.580,0.491,0.577
Microstates+LOF,0.530,20,0.458,0.546,0.558,0.478,0.628,0.747,0.483,0.476,0.598,0.550,0.568,0.522,0.462,0.676,0.391,0.477,0.530,0.369,0.507,0.584
ACM+TS+OCSVM,0.517,20,0.487,0.420,0.580,0.538,0.602,0.643,0.557,0.479,0.495,0.466,0.486,0.478,0.513,0.574,0.491,0.552,0.488,0.487,0.446,0.567


In [6]:
display_summary_by_dataset(
    experiment,
    mi_pipelines.keys(),
    ood_pipelines.keys(),
    experiment_specs=[
        ("MI BA", "mi", "MI_BA"),
        ("MI AUROC", "mi", "MI_AUROC"),
        ("OOD", "ood", "OOD_AUROC"),
    ],
)